In [9]:
import os

def organize_paths(root_dir):
    """
    Organizes file paths into a nested dictionary structured by consumer_id, model_name,
    search_strategy, and question.
    """
    organized_paths = {}

    for dirpath, dirnames, filenames in os.walk(root_dir):
        parts = dirpath.replace(root_dir, '').strip(os.sep).split(os.sep)
        if len(parts) >= 4:
            consumer_id = parts[0]
            model_name = parts[1]
            search_strategy = parts[2]
            question = parts[3]

            # Initialize dictionaries as needed
            organized_paths.setdefault(consumer_id, {})
            organized_paths[consumer_id].setdefault(model_name, {})
            organized_paths[consumer_id][model_name].setdefault(search_strategy, {})
            organized_paths[consumer_id][model_name][search_strategy].setdefault(question, [])

            for filename in filenames:
                file_path = os.path.join(dirpath, filename)
                organized_paths[consumer_id][model_name][search_strategy][question].append(file_path)

    return organized_paths

# Example usage:
    #root_directory = "/home/baptvit/repositories/graphrag-on-fhir/logs"
root_directory = "/home/baptvit/repositories/graphrag-on-fhir/logs"
paths_dict = organize_paths(root_directory)

    # # Print the organized paths
    # import pprint
    # pprint.pprint(paths_dict)

In [20]:
paths_dict["Edythe31_Morar593_9c3df38a-d3b7-2198-3898-51f9153d023d"]["gpt-4o-2024-08-06"]["similarity_search_0_hop"]["Q1:gpt-4o-2024-08-06:similarity_search_0_hop"]

['/home/baptvit/repositories/graphrag-on-fhir/logs/Edythe31_Morar593_9c3df38a-d3b7-2198-3898-51f9153d023d/gpt-4o-2024-08-06/similarity_search_0_hop/Q1:gpt-4o-2024-08-06:similarity_search_0_hop/output_step.json',
 '/home/baptvit/repositories/graphrag-on-fhir/logs/Edythe31_Morar593_9c3df38a-d3b7-2198-3898-51f9153d023d/gpt-4o-2024-08-06/similarity_search_0_hop/Q1:gpt-4o-2024-08-06:similarity_search_0_hop/tokens_step.json',
 '/home/baptvit/repositories/graphrag-on-fhir/logs/Edythe31_Morar593_9c3df38a-d3b7-2198-3898-51f9153d023d/gpt-4o-2024-08-06/similarity_search_0_hop/Q1:gpt-4o-2024-08-06:similarity_search_0_hop/search_step.json',
 '/home/baptvit/repositories/graphrag-on-fhir/logs/Edythe31_Morar593_9c3df38a-d3b7-2198-3898-51f9153d023d/gpt-4o-2024-08-06/similarity_search_0_hop/Q1:gpt-4o-2024-08-06:similarity_search_0_hop/tool_step.json']

## logic to merge the files

In [ ]:
BASE_DIR = "/home/baptvit/repositories/graphrag-on-fhir/logs"

OUTPUT_STEP = '{base_dir}/{consumer_id}/{experiment_number}/output_step.json'
SEARCH_STEP = '{base_dir}/{consumer_id}/{experiment_number}/search_step.json'
TOKEN_STEP = '{base_dir}/{consumer_id}/{experiment_number}/tokens_step.json'
TOOL_STEP = '{base_dir}/{consumer_id}/{experiment_number}/tool_step.json'

SAVE_INTERMEDIATE_DF = "/home/baptvit/repositories/graphrag-on-fhir/evaluations/data/silver/{consumer_id}.csv"

In [24]:
import pandas as pd

def generate_intermediate_dataframes(list_logs):

    for log_file in list_logs:
        if "output_step.json" in log_file:
            df_output_step = pd.read_json(log_file)
        elif "search_step.json" in log_file:
            df_search_step = pd.read_json(log_file)
        elif "tokens_step.json" in log_file:
            df_token_step = pd.read_json(log_file)
        elif "tool_step.json" in log_file:
            df_tool_step = pd.read_json(log_file)
        else:
            print("Not a valid log file: ", log_file)

    df_1 = df_output_step.merge(df_search_step, on='experiment_id', suffixes=('_output_step', '_search_step'))
    df_2 = df_token_step.merge(df_tool_step, on='experiment_id', suffixes=('_token_step', '_tool_step'))
    df_final = df_1.merge(df_2, on='experiment_id')  
    return df_final

In [25]:
list_test = paths_dict["Beatris270_Bogan287_5b3645de-a2d0-d016-0839-bab3757c4c58"]["gpt-4o-2024-08-06"]["similarity_search_0_hop"]["Q1:gpt-4o-2024-08-06:similarity_search_0_hop"]

In [26]:
df_gpt_test = generate_intermediate_dataframes(list_test)

In [27]:
df_gpt_test.head()

,experiment_id,timestamp_output_step,full_response,system_promt,input,output,timestamp_search_step,latency_s,query,similarity_threshold,...,timestamp_tool_step,records,caracteres_count,caracteres_count_after_reduce,pass_map_reduce,resource_type,main_keys,user_query,llm_token_limit,strategy_name
0,Q1:gpt-4o-2024-08-06:similarity_search_0_hop,2025-02-03 18:50:28,"{'input': ""What's my current medications and h...",\nAs an expert in interpreting Electronic Heal...,What's my current medications and how should I...,"Based on your electronic health records, here ...",2025-02-03 18:50:22,0.141131,Similarity Search 0-Hop,0.84,...,2025-02-03 18:50:22,{'Main health record': 'Resource Type: Medicat...,5195,5195,False,,medications,,128000,SimilaritySearch0HopStrategy


In [28]:
df_gpt_test.columns

Index(['experiment_id', 'timestamp_output_step', 'full_response',
       'system_promt', 'input', 'output', 'timestamp_search_step', 'latency_s',
       'query', 'similarity_threshold', 'k', 'embedding_model',
       'timestamp_token_step', 'total_cost', 'total_tokens',
       'successful_requests', 'completion_tokens', 'prompt_tokens',
       'total_time', 'timestamp_tool_step', 'records', 'caracteres_count',
       'caracteres_count_after_reduce', 'pass_map_reduce', 'resource_type',
       'main_keys', 'user_query', 'llm_token_limit', 'strategy_name'],
      dtype='object')

In [57]:
EVAL_DIR = "/home/baptvit/repositories/graphrag-on-fhir/evaluations/data/silver"
SAVE_INTERMEDIATE_DF = os.path.join(EVAL_DIR, "{consumer_id}-{model}.csv")

In [58]:
def run_per_consumer_id(str_consumer_id):
    root_directory = "/home/baptvit/repositories/graphrag-on-fhir/logs"
    paths_dict = organize_paths(root_directory)
    consumer_id = paths_dict[str_consumer_id]
    list_dataframes = []
    for model in list(consumer_id.keys()):
        list_dataframes_per_models = []
        searchs_strategies = consumer_id[model]
        for search_strategy in list(searchs_strategies.keys()):
            questions = consumer_id[model][search_strategy]
            for questions_exp in list(questions.keys()):
                paths_list = consumer_id[model][search_strategy][questions_exp]
                df_questions = generate_intermediate_dataframes(paths_list)
                list_dataframes_per_models.append(df_questions)
    df_concat = pd.concat(list_dataframes_per_models)
    df_concat.to_csv(SAVE_INTERMEDIATE_DF.format(consumer_id=str_consumer_id, model=model))

In [59]:
run_per_consumer_id("Milton509_Ortiz186_d66b5418-06cb-fc8a-8c13-85685b6ac939")

OSError: Cannot save file into a non-existent directory: '/home/baptvit/repositories/graphrag-on-fhir/evaluations/data/silver/Milton509_Ortiz186_d66b5418-06cb-fc8a-8c13-85685b6ac939'

In [53]:
df = pd.read_csv("/home/baptvit/repositories/graphrag-on-fhir/evaluations/data/silver/Milton509_Ortiz186_d66b5418-06cb-fc8a-8c13-85685b6ac939-gemini-1.5-pro.csv")

In [54]:
df.count()

Unnamed: 0                       47
experiment_id                    47
timestamp_output_step            47
full_response                    47
system_promt                     47
input                            47
output                           47
timestamp_search_step            47
latency_s                        47
query                            47
similarity_threshold             47
k                                47
embedding_model                  47
timestamp_token_step             47
total_cost                       47
total_tokens                     47
successful_requests              47
completion_tokens                47
prompt_tokens                    47
total_time                       47
timestamp_tool_step              47
records                          41
caracteres_count                 47
pass_map_reduce                  47
resource_type                    23
main_keys                        24
user_query                        0
llm_token_limit             

In [55]:
df = pd.read_csv("/home/baptvit/repositories/graphrag-on-fhir/evaluations/data/silver/Milton509_Ortiz186_d66b5418-06cb-fc8a-8c13-85685b6ac939.csv")

In [56]:
df.count()

Unnamed: 0               32
experiment_id            32
timestamp_output_step    32
full_response            32
system_promt             32
input                    32
output                   32
timestamp_search_step    32
latency_s                32
query                    32
similarity_threshold     32
k                        32
embedding_model          32
timestamp_token_step     32
total_cost               32
total_tokens             32
successful_requests      32
completion_tokens        32
prompt_tokens            32
total_time               32
timestamp_tool_step      32
records                  28
caracteres_count         32
pass_map_reduce          32
resource_type            16
main_keys                16
user_query                0
llm_token_limit          32
strategy_name            32
dtype: int64